# Tutorial 07: Data Generation Workflows

Learn how to generate custom synthetic PDPTW instances with control over problem characteristics.

**What you'll learn:**
- Generate synthetic maps with custom parameters
- Control demand patterns and distributions
- Create PDPTW instances with specific properties
- Scale problem size systematically

**Prerequisites:**
- Tutorial 01 (Quickstart)

**Time:** ~15 minutes

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import random

# Data generation
from vrp_toolkit.data.map import RealMap
from vrp_toolkit.data.generators import DemandGenerator, OrderGenerator

# Problem and solver
from vrp_toolkit.problems.pdptw import PDPTWInstance
from vrp_toolkit.algorithms.alns.solver import ALNSSolver, greedy_insertion_initial_solution

# Set seed
np.random.seed(42)
random.seed(42)

print("Ready for data generation!")

## 2. Quick Start: Generate Small Instance

Let's start with a small instance (2 restaurants, 4 customers):

In [ ]:
# Create map
real_map = RealMap(
    n_r=2,  # 2 restaurants
    n_c=4,  # 4 customers
    dist_function=np.random.uniform,
    dist_params={'low': -1, 'high': 1}
)

print(f"Map created: {real_map.N_R} restaurants, {real_map.N_C} customers")
print(f"Total nodes: {len(real_map.all_nodes)}")

In [ ]:
# Generate demands
random_params = {
    'sample_dist': {'function': np.random.randint, 'params': {'low': 1, 'high': 3}},
    'demand_dist': {'function': np.random.poisson, 'params': {'lam': 2}}
}

demands = DemandGenerator(
    time_range=30,
    time_step=10,
    restaurants=real_map.restaurants,
    customers=real_map.customers,
    random_params=random_params
)

print(f"Demands generated for {len(demands.time_intervals)} time intervals")
print(demands.demand_table.head())

In [ ]:
# Create orders
time_params = {
    'time_window_length': 30,
    'service_time': 5,
    'extra_time': 10
}

orders = OrderGenerator(
    real_map=real_map,
    demand_table=demands.demand_table,
    time_params=time_params,
    robot_speed=4
)

order_table = orders.get_order_table()
print(f"Orders created: {len(order_table)} rows")
print(order_table[['ID', 'Type', 'Demand', 'StartTime', 'EndTime']].head())

## 3. Scaling Problem Size

Generate instances of different sizes:

In [ ]:
def generate_instance(n_r, n_c, time_range=60, time_step=20):
    """Generate PDPTW instance with specified size."""
    # Map
    real_map = RealMap(
        n_r=n_r, n_c=n_c,
        dist_function=np.random.uniform,
        dist_params={'low': -1, 'high': 1}
    )
    
    # Demands
    random_params = {
        'sample_dist': {'function': np.random.randint, 'params': {'low': 1, 'high': n_r*n_c}},
        'demand_dist': {'function': np.random.poisson, 'params': {'lam': 2}}
    }
    
    demands = DemandGenerator(
        time_range=time_range,
        time_step=time_step,
        restaurants=real_map.restaurants,
        customers=real_map.customers,
        random_params=random_params
    )
    
    # Orders
    time_params = {'time_window_length': 30, 'service_time': 5, 'extra_time': 10}
    orders = OrderGenerator(real_map, demands.demand_table, time_params, robot_speed=4)
    
    order_table = orders.get_order_table()
    n_orders = len(order_table[order_table['Type'] == 'cp'])
    
    return order_table, real_map.distance_matrix, n_orders

# Test different sizes
sizes = [(2, 4), (3, 6), (4, 8)]
for n_r, n_c in sizes:
    order_table, dist_mat, n_orders = generate_instance(n_r, n_c)
    print(f"{n_r}R x {n_c}C: {n_orders} orders, {len(order_table)} total nodes")

## 4. Controlling Demand Patterns

### 4.1 Sparse vs Dense Demand

In [ ]:
real_map = RealMap(n_r=2, n_c=4, dist_function=np.random.uniform, dist_params={'low': -1, 'high': 1})

# Sparse demand (low lambda)
sparse_params = {
    'sample_dist': {'function': np.random.randint, 'params': {'low': 1, 'high': 3}},
    'demand_dist': {'function': np.random.poisson, 'params': {'lam': 1}}  # Low
}

sparse_demands = DemandGenerator(30, 10, real_map.restaurants, real_map.customers, sparse_params)
sparse_orders = OrderGenerator(real_map, sparse_demands.demand_table, 
                               {'time_window_length': 30, 'service_time': 5, 'extra_time': 10}, 4)
sparse_n = len(sparse_orders.get_order_table()[sparse_orders.get_order_table()['Type'] == 'cp'])

# Dense demand (high lambda)
dense_params = {
    'sample_dist': {'function': np.random.randint, 'params': {'low': 1, 'high': 3}},
    'demand_dist': {'function': np.random.poisson, 'params': {'lam': 5}}  # High
}

dense_demands = DemandGenerator(30, 10, real_map.restaurants, real_map.customers, dense_params)
dense_orders = OrderGenerator(real_map, dense_demands.demand_table,
                              {'time_window_length': 30, 'service_time': 5, 'extra_time': 10}, 4)
dense_n = len(dense_orders.get_order_table()[dense_orders.get_order_table()['Type'] == 'cp'])

print(f"Sparse demand (λ=1): {sparse_n} orders")
print(f"Dense demand (λ=5): {dense_n} orders")

### 4.2 Tight vs Loose Time Windows

In [ ]:
# Tight time windows (15 min)
tight_params = {'time_window_length': 15, 'service_time': 5, 'extra_time': 5}
tight_orders = OrderGenerator(real_map, demands.demand_table, tight_params, 4)

# Loose time windows (60 min)
loose_params = {'time_window_length': 60, 'service_time': 5, 'extra_time': 10}
loose_orders = OrderGenerator(real_map, demands.demand_table, loose_params, 4)

print("Tight windows (15 min):")
print(tight_orders.get_order_table()[['ID', 'Type', 'StartTime', 'EndTime']].head())
print("\nLoose windows (60 min):")
print(loose_orders.get_order_table()[['ID', 'Type', 'StartTime', 'EndTime']].head())

## 5. Generate and Solve

Create a complete instance and solve it:

In [ ]:
# Generate instance
order_table_final, dist_matrix, n_orders = generate_instance(2, 4, time_range=30, time_step=10)

# Create PDPTW instance
instance = PDPTWInstance(
    order_table=order_table_final,
    distance_matrix=dist_matrix,
    time_matrix=dist_matrix / 4.0,
    robot_speed=4.0
)

print(f"Instance: {instance.n} orders, {len(instance.indices)} nodes")
print(f"Distance matrix shape: {dist_matrix.shape}")
print(f"\\nInstance is ready to solve! Use ALNSSolver as shown in Tutorial 01.")

## 6. Summary

**What you learned:**
- ✅ Generate synthetic maps (RealMap)
- ✅ Control demand patterns (DemandGenerator)
- ✅ Create order tables (OrderGenerator)
- ✅ Scale problem size systematically
- ✅ Adjust time windows and service times

**Key parameters:**
- `n_r`, `n_c`: Number of restaurants and customers
- `lam` (Poisson): Controls order density
- `time_window_length`: Controls schedule flexibility
- `time_range`, `time_step`: Time granularity

**Next steps:**
- Try Tutorial 05 for sensitivity analysis
- Experiment with different distributions
- Create benchmark instances